# Kotodama Model A — private A7 cloud training

Before running: enable **GPU** and **Internet** in Kaggle Notebook settings. Add a **private, code-only** Kaggle Dataset named `kotodama-source`; it must not contain `.env`, JVS, learner recordings, manifests, or old artifacts. JVS is downloaded directly to `/tmp` and is never copied to `/kaggle/working`.

In [ ]:
from pathlib import Path
import shutil

INPUT_ROOT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/kotodama')
PRIVATE_ROOT = Path('/tmp/kotodama-private')

model_dirs = [
    path for path in INPUT_ROOT.rglob('model_a')
    if path.is_dir() and path.parent.name == 'ml'
]
if len(model_dirs) != 1:
    raise RuntimeError(f'Expected exactly one ml/model_a source tree under {INPUT_ROOT}; found: {model_dirs}')
source_root = model_dirs[0].parent.parent
if PROJECT.exists():
    shutil.rmtree(PROJECT)
shutil.copytree(source_root, PROJECT)
PRIVATE_ROOT.mkdir(parents=True, exist_ok=True)
print({'project': str(PROJECT), 'privateData': str(PRIVATE_ROOT)})

In [ ]:
%cd /kaggle/working/kotodama
!bash ml/model_a/cloud/bootstrap_gpu.sh /kaggle/working/kotodama

In [ ]:
import os
os.environ['KOTODAMA_ACCEPT_JVS_RESEARCH_TERMS'] = 'yes'
os.environ['KOTODAMA_DATA_ROOT'] = '/tmp/kotodama-private/jvs'
os.environ['KOTODAMA_MANIFEST_PATH'] = '/tmp/kotodama-private/manifests/jvs-a7-kaggle-v1.jsonl'
os.environ['KOTODAMA_RECORDINGS_ROOT'] = '/tmp/kotodama-private/recordings/jvs-a7-kaggle-v1'
os.environ['KOTODAMA_ARTIFACT_PATH'] = '/tmp/kotodama-private/artifacts/a7-jvs-ctc-kaggle-v1'
%cd /kaggle/working/kotodama
!bash ml/model_a/cloud/run_a7_jvs_cloud.sh /kaggle/working/kotodama

In [ ]:
import shutil
from pathlib import Path

artifact = Path('/tmp/kotodama-private/artifacts/a7-jvs-ctc-kaggle-v1')
assert (artifact / 'training-summary.json').is_file(), 'Training did not produce a complete artifact.'
output = shutil.make_archive('/kaggle/working/a7-jvs-ctc-kaggle-v1', 'zip', artifact)
print('Download this private output only:', output)